# 04 — Обучение U-Net

Главная модель проекта. Архитектура и функции потерь — в `src/models.py`.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import callbacks

from src import config, models, visualization, metrics

print('GPU доступен:', tf.config.list_physical_devices('GPU'))
# Если нет GPU — обучение займёт 2-6 часов на CPU. Это нормально.


In [ ]:
X_train = np.load(config.DATA_PATCHES / 'X_train.npy').astype(np.float32)
X_val   = np.load(config.DATA_PATCHES / 'X_val.npy').astype(np.float32)
X_test  = np.load(config.DATA_PATCHES / 'X_test.npy').astype(np.float32)
y_train = np.load(config.DATA_PATCHES / 'y_train.npy')
y_val   = np.load(config.DATA_PATCHES / 'y_val.npy')
y_test  = np.load(config.DATA_PATCHES / 'y_test.npy')

N_CHANNELS = X_train.shape[-1]
unet = models.build_unet(input_shape=(config.PATCH_SIZE, config.PATCH_SIZE, N_CHANNELS))
unet = models.compile_model(unet)
unet.summary()


In [ ]:
GlacierDataGenerator = models.build_data_generator()

train_gen = GlacierDataGenerator(X_train, y_train, batch_size=config.BATCH_SIZE, augment=True)
val_gen   = GlacierDataGenerator(X_val,   y_val,   batch_size=config.BATCH_SIZE, augment=False)

config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cbs = [
    callbacks.ModelCheckpoint(
        str(config.MODELS_DIR / 'unet_best.h5'),
        save_best_only=True, monitor='val_dice_coefficient', mode='max', verbose=1,
    ),
    callbacks.EarlyStopping(
        patience=15, restore_best_weights=True, monitor='val_dice_coefficient', mode='max',
    ),
    callbacks.ReduceLROnPlateau(
        factor=0.5, patience=7, monitor='val_loss', min_lr=1e-6, verbose=1,
    ),
    callbacks.CSVLogger(str(config.RESULTS_DIR / 'training_log.csv')),
]


In [ ]:
history = unet.fit(
    train_gen,
    validation_data=val_gen,
    epochs=config.EPOCHS,
    callbacks=cbs,
    verbose=1,
)


In [ ]:
visualization.plot_training_curves(history, save_path=config.FIGURES_DIR / 'training_curves.png')


## Оценка на тестовой выборке

In [ ]:
y_pred_prob = unet.predict(X_test, batch_size=config.BATCH_SIZE)
y_pred = (y_pred_prob.squeeze() > 0.5).astype(int)

unet_metrics = metrics.evaluate_segmentation(y_test, y_pred)
print('U-Net результаты на тестовой выборке:')
for k, v in unet_metrics.items():
    print(f'  {k}: {v:.3f}')

visualization.plot_prediction_grid(X_test, y_test, y_pred, n_examples=4,
                                    save_path=config.FIGURES_DIR / 'predictions_examples.png')


## Обновление сравнительной таблицы

In [ ]:
df_results = pd.read_csv(config.TABLES_DIR / 'model_comparison.csv')
df_results = pd.concat([df_results, pd.DataFrame([{
    'Метод': 'U-Net (наш)',
    'F1-score': unet_metrics['f1'],
    'Precision': unet_metrics['precision'],
    'Recall': unet_metrics['recall'],
    'IoU': unet_metrics['iou'],
}])], ignore_index=True)

print(df_results.to_string(index=False))
df_results.to_csv(config.TABLES_DIR / 'model_comparison.csv', index=False)

visualization.plot_model_comparison(df_results, metric='F1-score',
                                     save_path=config.FIGURES_DIR / 'model_comparison_f1.png')
visualization.plot_model_comparison(df_results, metric='IoU',
                                     save_path=config.FIGURES_DIR / 'model_comparison_iou.png')
